In [ ]:
import numpy as np
import torch
from torch import nn
import matplotlib as mpl
import matplotlib.pyplot as plt
import sys
import os

# System definitions

In [ ]:
from nsflows.systems.lennard_jones import lennard_jones
from nsflows.systems.uniforms import box_uniform
from nsflows.tools.util import density_from_box_length

n_particles = 8
dimensions = 2
cutin = 0.8

# State points provided with this repository, keyed by the density as reported in
# the paper. The box length is the exact value the simulations were run at, and is
# what names the data directories under data/lj/; the key is that density rounded
# to two decimals. Change `density` to switch between state points: everything
# below, including which initial samples are loaded, follows from it.
box_length_for_density = {
    0.95: 2.9,
    0.73: 3.3,
}

density = 0.95

if density not in box_length_for_density:
    raise KeyError(
        f"No data provided for density={density}; "
        f"available: {sorted(box_length_for_density)}"
    )

box_length = box_length_for_density[density]
rho = density_from_box_length(box_length, n_particles, dimensions)

print(f"Selected density {density}: box length {box_length}, exact density {rho:.6f}")

if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print(f"Using device: {device}")

box_uniform_2D = box_uniform(n_particles=n_particles, dimensions=dimensions, device=device, box_length=box_length)
LJ_disks = lennard_jones(n_particles=n_particles, dimensions=dimensions, rho=rho, device=device, cutin=cutin, lrc=True)

## Define Parameters

In [ ]:
# Load Previous Training
load = False

# Nested Sampling Parameters
live_samples = 10000

# The initial live-sample set is picked from data/lj/ according to live_samples and
# the state point selected above.
init_samples_dir = "../data/lj"
init_samples_filepath = os.path.join(init_samples_dir, f"K{live_samples:05d}", f"L{box_length}", "samples_init.pt")
if not os.path.exists(init_samples_filepath):
    raise FileNotFoundError(
        f"No initial samples shipped for live_samples={live_samples} at box length "
        f"{box_length} (looked for {init_samples_filepath})"
    )

max_ns_iterations = 500000
n_propagate = 1000
update_step = True
turn_on_nf = 100
alternate_std_ns_iters = 100
n_pool = 20000
load_nf_parameters = False
itrain = 0 # Switch between fine tuning and full training
reinitialize_nf_parameters = False
cumulate_n_dataset = 3

# Network Parameters
conditioned = True
n_blocks = 28
n_bins = 12

max_lr = 2.5e-4
mid_lr = 1e-4 
start_lr = max_lr/25
end_lr = 1e-6

# Training Parameters
training_protocol = [
    # {
    #     "w_xz" : 1,
    #     "w_zx" : 0,
    #     "batch_size" : 1000,
    #     "conds_per_batch" : -1,
    #     "total_steps" : 750,
    #     "start_lr" : start_lr,
    #     "end_lr" : mid_lr,
    #     "max_lr" : max_lr,
    #     "save_best" : True,
    #     "optimizer" : "Adam",
    #     "scheduler" : "OneCycleLR",
    # }, 

    {
        "w_xz" : 1,
        "w_zx" : 0,
        "batch_size" : 1000,
        "conds_per_batch" : -1,
        "total_steps" : 250,
        "start_lr" : mid_lr,
        "end_lr" : end_lr,
        "max_lr" : None,
        "save_best" : True,
        "optimizer" : "Adam",
        "scheduler" : "CosineAnnealingLR",
    },       
]


# Output Folder Definition

In [ ]:
from nsflows.tools.util import generate_unique_identifier, remove_empty_directories, generate_output_directory

root_folder = "./output/L2.9/"
remove_empty_directories(root_folder=root_folder)

if load:
    output_dir = "./cluster_run/L2.9/FT"
    print(f"Run Folder: {output_dir}")
else:
    run_id = generate_unique_identifier()
    output_dir = generate_output_directory(run_id, root_folder=root_folder)

In [ ]:
# ============================================================
# Generate summary for new runs
# OR
# Read and display existing summary for loaded runs
# ============================================================

from pathlib import Path
from datetime import datetime
import json

summary_txt_path = Path(output_dir) / "simulation_summary.txt"
summary_json_path = Path(output_dir) / "simulation_summary.json"

if not load:

    # --------------------------------------------------------
    # Build parameter summary dictionary
    # --------------------------------------------------------

    summary = {
        "timestamp": datetime.now().isoformat(),

        "system": {
            "n_particles": n_particles,
            "dimensions": dimensions,
            "box_length": box_length,
            "cutin": cutin,
            "rho": rho,
            "device": str(device),
        },

        "nested_sampling": {
            "init_samples_filepath": init_samples_filepath,
            "live_samples": live_samples,
            "max_ns_iterations": max_ns_iterations,
            "n_propagate": n_propagate,
            "update_step": update_step,
            "turn_on_nf": turn_on_nf,
            "alternate_std_ns_iters": alternate_std_ns_iters,
            "n_pool": n_pool,
            "load_nf_parameters": load_nf_parameters,
            "itrain": itrain,
            "reinitialize_nf_parameters": reinitialize_nf_parameters,
            "cumulate_n_dataset": cumulate_n_dataset,
        },

        "network": {
            "conditioned": conditioned,
            "n_blocks": n_blocks,
            "n_bins": n_bins,
        },

        "learning_rates": {
            "start_lr": start_lr,
            "mid_lr": mid_lr,
            "max_lr": max_lr,
            "end_lr": end_lr,
        },

        "training_protocol": training_protocol,
    }

    # --------------------------------------------------------
    # Ensure output directory exists
    # --------------------------------------------------------

    Path(output_dir).mkdir(parents=True, exist_ok=True)

    # --------------------------------------------------------
    # Save JSON summary
    # --------------------------------------------------------

    with open(summary_json_path, "w") as f:
        json.dump(summary, f, indent=4)

    # --------------------------------------------------------
    # Save readable text summary
    # --------------------------------------------------------

    with open(summary_txt_path, "w") as f:

        f.write("====================================================\n")
        f.write("Simulation Summary\n")
        f.write("====================================================\n\n")

        f.write(f"Generated: {summary['timestamp']}\n\n")

        f.write("SYSTEM PARAMETERS\n")
        f.write("-----------------\n")
        for k, v in summary["system"].items():
            f.write(f"{k}: {v}\n")

        f.write("\nNESTED SAMPLING PARAMETERS\n")
        f.write("--------------------------\n")
        for k, v in summary["nested_sampling"].items():
            f.write(f"{k}: {v}\n")

        f.write("\nNETWORK PARAMETERS\n")
        f.write("------------------\n")
        for k, v in summary["network"].items():
            f.write(f"{k}: {v}\n")

        f.write("\nLEARNING RATES\n")
        f.write("--------------\n")
        for k, v in summary["learning_rates"].items():
            f.write(f"{k}: {v}\n")

        f.write("\nTRAINING PROTOCOL\n")
        f.write("-----------------\n")

        for i, protocol in enumerate(training_protocol):
            f.write(f"\nStage {i+1}\n")
            f.write("~~~~~~~~~~~~\n")
            for k, v in protocol.items():
                f.write(f"{k}: {v}\n")

    print(f"Summary files written to:\n")
    print(f"  JSON : {summary_json_path}")
    print(f"  TEXT : {summary_txt_path}")

else:

    # --------------------------------------------------------
    # Read and print existing summary
    # --------------------------------------------------------

    try:
        with open(summary_txt_path, "r") as f:
            summary_contents = f.read()

        print("====================================================")
        print("Loaded Run Summary")
        print("====================================================\n")

        print(summary_contents)

    except FileNotFoundError:
        print("WARNING: No simulation summary file found.")
        print(f"Expected location:\n{summary_txt_path}")

    except Exception as e:
        print("ERROR while reading simulation summary:")
        print(e)

# Flow definition

In [ ]:
import copy

from nsflows.network.flow_assembler import flow_assembler
from nsflows.network.circular_shift import circular_shift
from nsflows.network.coupling_blocks import EquivariantRQS
from nsflows.transformations.normalization import NormalizeBox
from nsflows.transformations.remove_origin import remove_origin

block_unit = [
    
    circular_shift(n_particles-1, dimensions, device),
    EquivariantRQS((0,), n_particles-1, dimensions, device, left=-1, bottom=-1, right=1, top=1, n_bins=n_bins, conditioned=conditioned),
    EquivariantRQS((1,), n_particles-1, dimensions, device, left=-1, bottom=-1, right=1, top=1, n_bins=n_bins, conditioned=conditioned),
    
    circular_shift(n_particles-1, dimensions, device),
    EquivariantRQS((1,), n_particles-1, dimensions, device, left=-1, bottom=-1, right=1, top=1, n_bins=n_bins, conditioned=conditioned),
    EquivariantRQS((0,), n_particles-1, dimensions, device, left=-1, bottom=-1, right=1, top=1, n_bins=n_bins, conditioned=conditioned),
]
block_list = [copy.deepcopy(element) for block in range(n_blocks) for element in block_unit]

box_pr = torch.from_numpy(np.array([box_length, box_length], dtype=np.float32)).to(device)
box_sys = torch.from_numpy(np.array([box_length, box_length], dtype=np.float32)).to(device)

norm_box_pr = NormalizeBox(n_particles=n_particles, dimensions=dimensions, box_length=box_pr, device=device)
norm_box_sys = NormalizeBox(n_particles=n_particles, dimensions=dimensions, box_length=box_sys, device=device)
rm_origin = remove_origin(n_particles=n_particles, dimensions=dimensions, device=device)

# Flow Initialization  
flow = flow_assembler(box_uniform_2D, LJ_disks, device=device, 
                    blocks = block_list,
                    prior_sided_transformation_layers = [norm_box_pr, rm_origin], 
                    post_sided_transformation_layers = [norm_box_sys, rm_origin]
                    ).to(device)

flow_parameters = sum(p.numel() for p in flow.parameters() if p.requires_grad)
print(f"Network parameters: {flow_parameters}")

# Nested Sampling

In [ ]:
import os
import re
import glob

from nsflows.samplers.monte_carlo import rejection_monte_carlo
from nsflows.samplers.DL_samplers import nflows_propagator
from nsflows.nested_sampling import nested_sampling

rejection_sampler = rejection_monte_carlo(system=LJ_disks, n_cycles=100, step_size=1.2, transform=True)
nflows_sampler = nflows_propagator(flow, conditioned, transform=True)

if load:
    acceptance, umax_plt = np.loadtxt(os.path.join(output_dir, "output.txt"), usecols=(2,3), unpack=True)
    samples = torch.load(os.path.join(output_dir, "samples.pt"))
    U_samples = umax_plt[-1]

    save_best = training_protocol[-1]["save_best"]

    # Base name depending on protocol
    prefix = "best_flow_parameters" if save_best else "flow_parameters"

    # Regex patterns for:
    #   prefix_<count>.pt
    #   prefix_<count>_<stage>.pt
    patterns = [
        re.compile(rf"{prefix}_(\d+)\.pt$"),
        re.compile(rf"{prefix}_(\d+)_(\d+)\.pt$"),
    ]

    loaded_files = []

    # Look for all candidates
    for filepath in sorted(glob.glob(os.path.join(output_dir, f"{prefix}_*.pt"))):
        filename = os.path.basename(filepath)
        count = stage = None

        # Try matching allowed formats
        for pat in patterns:
            m = pat.match(filename)
            if m:
                groups = m.groups()
                if len(groups) == 1:
                    count = int(groups[0])
                elif len(groups) == 2:
                    count, stage = map(int, groups)
                break

        if not m:
            continue  # skip unexpected filenames

        print(f"Loading network parameters from {filepath}")
        loaded_files.append({
            "filepath": filepath,
            "count": count,
            "stage": stage,
        })

    print(f"Found {len(loaded_files)} parameter files.")
    nflows_sampler.flow.load_state_dict(torch.load(loaded_files[-1]["filepath"]))
else:
    samples, U_samples, acceptance, umax_plt = nested_sampling(K=live_samples, 
                                                            system=LJ_disks, 
                                                            std_propagator=rejection_sampler, 
                                                            nf_propagator=nflows_sampler, 
                                                            init_samples_filepath=init_samples_filepath, 
                                                            max_iters=max_ns_iterations, 
                                                            n_propagate=n_propagate, 
                                                            update_step=update_step,
                                                            turn_on_nf=turn_on_nf, 
                                                            alternate_std_ns_iters=alternate_std_ns_iters, 
                                                            n_pool=n_pool, 
                                                            load_nf_parameters=load_nf_parameters, 
                                                            reinitialize_nf_parameters=reinitialize_nf_parameters,
                                                            itrain=itrain,
                                                            cumulate_n_dataset=cumulate_n_dataset,
                                                            training_protocol=training_protocol,
                                                            iprint=1, 
                                                            isavesamp=500,
                                                            save_biased_pool=False, 
                                                            outputdir=output_dir,
                                                            disable_pbar=True)

# Plot Output

## Training Metrics

In [ ]:
import re
import glob

# Regex patterns for possible filename formats
patterns = [
    re.compile(r"train_log_(\d+)\.txt$"),               # train_log_count.txt OR train_log_stage.txt
    re.compile(r"train_log_(\d+)_(\d+)\.txt$"),         # train_log_count_stage.txt
]

all_metrics = []

# Find all candidate files
for filepath in sorted(glob.glob(os.path.join(output_dir, "train_log_*.txt"))):
    filename = os.path.basename(filepath)
    stage = count = None

    # Try matching each pattern
    for pat in patterns:
        match = pat.match(filename)
        if match:
            groups = match.groups()
            if len(groups) == 1:
                # Could be count-only or stage-only — label generically
                count = int(groups[0])
            elif len(groups) == 2:
                count, stage = map(int, groups)
            break

    if not match:
        # Skip anything that doesn’t match expected formats
        continue

    try:
        print(f"Loaded: {filename} (count={count}, stage={stage})")
        metrics = np.loadtxt(filepath)
    except Exception as e:
        print(f"Could not load {filename}: {e}")

    length_x = 60
    fig_size = (length_x * 0.393701, 10 * 0.393701)
    fig, ax = plt.subplots(1, 5, figsize = fig_size, dpi = 400, tight_layout=True)

    ax[0].plot(metrics[:,0], metrics[:,2], label="train")
    ax[0].plot(metrics[:,0], metrics[:,6], label="eval")
    ax[0].set_xlabel("epochs")
    ax[0].set_ylabel("NLL loss")

    ax[1].plot(metrics[:,0], metrics[:,3], label="train")
    ax[1].plot(metrics[:,0], metrics[:,7], label="eval")
    ax[1].set_xlabel("epochs")
    ax[1].set_ylabel("ECUT loss")

    ax[2].plot(metrics[:,0], metrics[:,4], label="train")
    ax[2].plot(metrics[:,0], metrics[:,8], label="eval")
    ax[2].set_xlabel("epochs")
    ax[2].set_ylabel("ECUT violation")
    ax[2].legend(frameon=False)

    ax[3].plot(metrics[:,0], metrics[:,5], color="C0", label="train")
    ax[3].set_xlabel("epochs")
    ax[3].set_ylabel("Gradient Norm")
    ax[3].legend(frameon=False)
    ax[3].set_yscale("log")

    ax[4].plot(metrics[:,0], metrics[:,9], color="C1", label="eval")
    ax[4].set_xlabel("epochs")
    ax[4].set_ylabel("RESS")
    ax[4].set_ylim(0, 1)
    ax[4].legend(frameon=False)

    plt.savefig(os.path.join(output_dir, f"metrics_{count:04d}.png"))
    plt.close()
    # plt.show()


## Reference, Generated, Resampled

In [ ]:
# Helpers for comparing configurations up to the symmetries of the system: a pi/2
# rotation of the box and a relabelling of the particles. They live in
# nsflows.tools.util so that the notebooks and the analysis scripts share one
# implementation.
from nsflows.tools.util import (
    remove_outermost_particle,
    align_config,
)


In [ ]:
# Reference configuration for the alignment: the deepest live set of the completed
# run shipped for this state point, with its outermost particle removed. Loaded once,
# outside the loop. Set align = False to plot the raw configurations instead.
align = True

if align:
    ref_filepath = os.path.join(init_samples_dir, f"K{live_samples:05d}", f"L{box_length}", "samples_ref.pt")
    if not os.path.exists(ref_filepath):
        raise FileNotFoundError(
            f"No reference configuration shipped for box length {box_length} "
            f"(looked for {ref_filepath}); set align = False to skip the alignment"
        )
    reff_config = torch.load(ref_filepath, map_location=device)
    refff_config = remove_outermost_particle(reff_config.view(-1, LJ_disks.n_particles, LJ_disks.dimensions))
    ref_config = refff_config[0].clone().unsqueeze(0)

count = 0
while True:
    
    dataset_filepath = os.path.join(output_dir, f"dataset_{count:04d}.pt")
    pool_biased_filepath = os.path.join(output_dir, f"pool_biased_{count:04d}.pt")
    pool_filepath = os.path.join(output_dir, f"pool_{count:04d}.pt")
    conds_filepath = os.path.join(output_dir, f"conds_{count:04d}.pt")
    if (not os.path.exists(dataset_filepath)) or (not os.path.exists(pool_biased_filepath)) or (not os.path.exists(pool_filepath)) or (not os.path.exists(conds_filepath)) :
        print(f"File {dataset_filepath} " + (f"found" if os.path.exists(dataset_filepath) else f"not found"))
        print(f"File {pool_biased_filepath} " + (f"found" if os.path.exists(pool_biased_filepath) else f"not found"))
        print(f"File {pool_filepath} " + (f"found" if os.path.exists(pool_filepath) else f"not found"))
        print(f"File {conds_filepath} " + (f"found" if os.path.exists(conds_filepath) else f"not found"))
        break
    count += 1

    dataset = torch.load(dataset_filepath)
    pool_biased = torch.load(pool_biased_filepath)
    pool = torch.load(pool_filepath)
    cond = torch.load(conds_filepath)

    # Align every configuration to a common reference, up to a pi/2 rotation of the
    # box and a relabelling of the particles, so the three panels can be compared
    # directly. Set align = False above to plot the raw configurations instead.
    if align:
        adataset = align_config(dataset.view(-1, LJ_disks.n_particles, LJ_disks.dimensions), ref_config, n_particles, dimensions, box_length)
        apool_biased = align_config(pool_biased.view(-1, LJ_disks.n_particles, LJ_disks.dimensions), ref_config, n_particles, dimensions, box_length)
        apool = align_config(pool.view(-1, LJ_disks.n_particles, LJ_disks.dimensions), ref_config, n_particles, dimensions, box_length)

        dataset_cpu = adataset.cpu().numpy()
        pool_biased_cpu = apool_biased.cpu().numpy()
        pool_cpu = apool.cpu().numpy()

    else:
        dataset_cpu = dataset.view(-1, LJ_disks.n_particles, LJ_disks.dimensions).cpu().numpy()
        pool_biased_cpu = pool_biased.view(-1, LJ_disks.n_particles, LJ_disks.dimensions).cpu().numpy()
        pool_cpu = pool.view(-1, LJ_disks.n_particles, LJ_disks.dimensions).cpu().numpy()

    fig_size = (30 * 0.393701, 10 * 0.393701)
    fig, ax = plt.subplots(1, 3, figsize = fig_size, dpi = 400, tight_layout=True, sharey=True)

    ax[0].set_aspect('equal')
    ax[0].scatter(dataset_cpu[:,:,0], dataset_cpu[:,:,1], s=.25, zorder = 10, alpha=0.04, color="C0")
    ax[0].set_xlim(-box_length/2, box_length/2)
    ax[0].set_ylim(-box_length/2, box_length/2)
    ax[0].set_xticks([-box_length/2, 0, box_length/2])
    ax[0].set_xticklabels([r"$-L/2$", r"$0$", r"$L/2$"])
    ax[0].set_yticks([-box_length/2, 0, box_length/2])
    ax[0].set_yticklabels([r"$-L/2$", r"$0$", r"$L/2$"])

    ax[1].set_aspect('equal')
    ax[1].scatter(pool_biased_cpu[:,:,0], pool_biased_cpu[:,:,1], s=.25, zorder = 10, alpha=0.04, color="C2")
    ax[1].set_xlim(-box_length/2, box_length/2)
    ax[1].set_ylim(-box_length/2, box_length/2)
    ax[1].set_xticks([-box_length/2, 0, box_length/2])
    ax[1].set_xticklabels([r"$-L/2$", r"$0$", r"$L/2$"])
    ax[1].set_yticks([-box_length/2, 0, box_length/2])
    ax[1].set_yticklabels([r"$-L/2$", r"$0$", r"$L/2$"])

    ax[2].set_aspect('equal')
    ax[2].scatter(pool_cpu[:,:,0], pool_cpu[:,:,1], s=.25, zorder = 10, alpha=0.04, color="C3")
    ax[2].set_xlim(-box_length/2, box_length/2)
    ax[2].set_ylim(-box_length/2, box_length/2)
    ax[2].set_xticks([-box_length/2, 0, box_length/2])
    ax[2].set_xticklabels([r"$-L/2$", r"$0$", r"$L/2$"])
    ax[2].set_yticks([-box_length/2, 0, box_length/2])
    ax[2].set_yticklabels([r"$-L/2$", r"$0$", r"$L/2$"])

    plt.savefig(os.path.join(output_dir, f"pools_noalign_{count:04d}.png"))
    plt.close()
    # plt.show()


## Generation Efficiency

In [ ]:
gen_attempts = []
count = 0 
while True:
    generation_log_filepath = os.path.join(output_dir, f"generation_log_{count:04d}.txt")
    if not os.path.exists(generation_log_filepath):
        print(f"File {generation_log_filepath} not found")
        break
    count += 1
    gen_attempts.append(np.loadtxt(generation_log_filepath, usecols=(0), unpack=True)[-1])

fig_size = (10 * 0.393701, 5 * 0.393701)
fig, ax = plt.subplots(figsize = fig_size, dpi = 400)

ax.set_yscale("log")
ax.set_ylim(4,9.9e3)
ax.plot(np.array(gen_attempts) + 1, "^", mfc="none", label=r"Full training")

ax.set_xlabel(r"Pool generated")
ax.set_ylabel(r"Generation attempts")

plt.savefig(os.path.join(output_dir, f"efficiency_{count:04d}.png"))
plt.show()

## Radial Distribution Functions

In [ ]:
from nsflows.tools.observables import rdf

gofrs = [] 
count = 0 
while True:
    pool_filepath = os.path.join(output_dir, f"pool_{count:04d}.pt")
    if not os.path.exists(pool_filepath):
        print(f"File {pool_filepath} not found")
        break
    count += 1

    pool = torch.load(pool_filepath)
    r, gofr = rdf(pool, n_particles=LJ_disks.n_particles, dimensions=LJ_disks.dimensions, box_length=LJ_disks.box_length)
    gofrs.append(gofr)

fig_size = (10 * 0.393701, 7.5 * 0.393701)
fig, ax = plt.subplots(figsize = fig_size, dpi = 400)

for gofr in gofrs[:10]:

    plt.plot(r, gofr)

plt.axhline(1, color="k", ls=":")
plt.savefig(os.path.join(output_dir, f"rdfs.png"))
plt.show()

## Energy vs. Iteration and Density of States vs. Iteration

In [ ]:
vals, bins = np.histogram(umax_plt-umax_plt.min(), bins=200)
bin_centers = .5*(bins[1:] + bins[:-1])

# Reference standard nested sampling run for this state point, drawn for comparison
# where one is shipped. Only L2.9 has one, so at other state points this run is
# plotted on its own. np.loadtxt reads the gzipped file directly.
reference_filepath = os.path.join(init_samples_dir, f"K{live_samples:05d}", f"L{box_length}", "reference_from_std_ns.txt.gz")
has_reference = os.path.exists(reference_filepath)

if has_reference:
    umax_plt_std = np.loadtxt(reference_filepath, unpack=True, usecols=3)
    vals_std, bins_std = np.histogram(umax_plt_std-umax_plt_std.min(), bins=200)
    bin_centers_std = .5*(bins_std[1:] + bins_std[:-1])
    iters_std = np.arange(len(umax_plt_std))/1e4
else:
    print(f"No standard nested sampling reference for box length {box_length}; plotting this run only.")

from mpl_toolkits.axes_grid1.inset_locator import inset_axes

# umax_plt covers the iterations that actually ran, which is fewer than
# max_ns_iterations if the run was stopped early.
iters = np.arange(len(umax_plt))/1e4

# Horizontal guides at a fixed number of evenly spaced energy levels.
n_guide_lines = 50
guide_step = max(1, len(umax_plt)//n_guide_lines)

fig_size = (10 * 0.393701, 7 * 0.393701)
fig, ax = plt.subplots(1, 2, figsize = fig_size, dpi = 400, sharey=True, tight_layout=True)

if has_reference:
    ax[0].plot(iters_std, (umax_plt_std-umax_plt_std.min()), color="C0", ls=":", label="std. NS")
ax[0].plot(iters, (umax_plt-umax_plt.min()), color="C3", label="NFs NS")
for u in ((umax_plt[guide_step//2::guide_step]-umax_plt.min())):

    ax[0].axhline(u, ls=":", lw=0.5, color="k")

if has_reference:
    ax[0].legend(loc="lower center", bbox_to_anchor=(0.5, 1.02), ncol=2, frameon=False)

ax[0].set_ylabel("Energy")
ax[0].set_xlabel(r"Iteration ($\times 10^4$)")
ax[0].set_xticks([0, max(iters[-1], iters_std[-1]) if has_reference else iters[-1]])

ax_inset = inset_axes(ax[0], width="50%", height="30%", loc='upper right', borderpad=.5)
if has_reference:
    ax_inset.plot(iters_std, (umax_plt_std-umax_plt_std.min()), color="C0", ls=":")
ax_inset.plot(iters, (umax_plt-umax_plt.min()), color="C3")
for u in ((umax_plt[guide_step//2::guide_step]-umax_plt.min())):

    ax_inset.axhline(u, ls=":", lw=0.5, color="k")
ax_inset.set_yscale("log")
ax_inset.set_ylim(4.0e-2,1.e3)

if has_reference:
    ax[1].plot(vals_std, bin_centers_std, color="C4", ls=":")
ax[1].plot(vals, bin_centers, color="C1")
ax[1].set_xlabel(r"Samples")
ax[1].set_xscale('log')

plt.savefig(os.path.join(output_dir, f"evsiter.png"))
plt.show()

## Timings

In [ ]:
data = np.genfromtxt(os.path.join(output_dir, "timings.txt"), skip_header=2, names=True, comments="#")
timings = np.column_stack([data[name] for name in data.dtype.names])

# Labels for tasks
labels = data.dtype.names

fig_size = (15 * 0.393701, 7 * 0.393701)
fig, ax = plt.subplots(figsize = fig_size, dpi = 400, sharey=True, tight_layout=True)

x = np.arange(timings.shape[0])  # one bar per row
bottom = np.zeros(timings.shape[0])

labels = [r"Standard MCMC ($10^2$ NS steps)", r"Selection from Pool (~ $10^4$ NS steps)", "Network Training", "Pool Generation"]
for i, label in enumerate(labels[2:]):
    ax.bar(x, timings[:, i+2], bottom=bottom, label=label, color=f"C{i+2}")
    bottom += np.nan_to_num(timings[:, i+2])  # accumulate heights, treating NaN as 0

ax.set_xlabel("Pool generated")
ax.set_ylabel("Time (seconds)")
# ax.set_title("Task timings per sample")
# ax.legend(title="Tasks")

ax.legend(loc='lower center',
                    bbox_to_anchor=(0.5, 1.02), ncol=2, frameon=False)

plt.tight_layout()
plt.savefig(os.path.join(output_dir, f"timings.png"))
plt.show()